#### Agents



##### First understand the probelm.
- suppose we build a normal chain : user-->Prompt-->LLM--> output. This workflow is fixed , no matter what the question is , the same steps will excecute. 
- example: if user asks ```what is spark``` the flow will be prompt-->llm-->Answer. this is good 
now if  user asks ```what is the weather in banglore today``` to answer this we need weather API.
- similarly if user asks ```what is 500 * 800``` we need calculator. and also similarly if the user asks ```how many customers purchased product A yesterday?``` here we need sql database tool.
- The question is here who decides which tool to use, the answer is ``Agent``.

- An agent is a LLM-Powered system that can reason, decide which tools to use, excecute them, observe the results, and continue reaches until a final answer.
- think like 
```
Question
   ↓
Reason
   ↓
Choose Tool
   ↓
Execute Tool
   ↓
Observe Result
   ↓
Repeat if needed
   ↓
Final Answer
```

#### What is an Agnet ?
- in a chain sequence of steps is fixed , here we decide to code time what happens in what order. 
- in an agent The llm decides at a runtime which tools to call, in what order,and when to stop
```
Chain:  User → Step1 → Step2 → Step3 → Answer   ← YOU decide the steps
Agent:  User → LLM thinks → calls Tool A → reads result → 
               LLM thinks → calls Tool B → reads result →
               LLM thinks → "I have enough, here's the answer"  ← LLM decides
```
- The key mental model: The LLm is the brain, tools are the hands.


#### Real world analogy
- imagine a manager. workers are calculate worker , sql worker, search worker, email worker. and manager: will decide who should these tasks ?, in what order? how many times?. here the manager is the agent , the workers are tools.

#### Chain vs Agent 
- chain : Fixed sequence   ```prompt-->llm-->output. this alsways same
- agent : dynamic sequence ```question-->think-->observe-->think again-->another tool-->answer

#### Agent loop
- Every agent runs a loop called ```ReAct--- Reasoning + Acting.```
- thought-->action-->observation-->though-->action-->observation-->Final Answer. this cycle is called reasoning loop.

```
Thought:  "The user wants to know sales by region. I should query the database."
Action:   get_regional_sales(year=2024)
Observation: "North: ₹4.5M, South: ₹3.2M, East: ₹2.8M, West: ₹5.1M"

Thought:  "Now I know the regional breakdown. West is highest. I should also get the growth rate."
Action:   get_growth_rate(region="West", year=2024)
Observation: "West region grew 34% YoY"

Thought:  "I have all the information needed to answer."
Final Answer: "West leads with ₹5.1M in sales and 34% YoY growth..."
```
- This loop repeats unitl the llm has enough information to answer.

#### Components of Agent
every agent has the following components.
```
LLM
Tools
Prompt
Memory (optional)
Output Parser
Executor
```
                Tools
                  ↑
                  |
User → Agent → LLM
                  |
                  ↓
            Final Answer
```

#### Agent Excecutor
- here the agent decides. and agent excecutor Excecutes 
- example :
```
agent_excecutor.invoke({"input":"what is the weather?"})
```
internally
```
Agent
 ↓
Tool Selection
 ↓
Tool Execution
 ↓
Final Answer
``` 

Level 1 — our First Agent


In [18]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
load_dotenv(override=True)

# step 1: define tool
@tool
def get_weather(city:str)->str:
    """
    Get the current weather for an city.
    use when users asks about weather, temperature, or climate
    """
    weather_data = {
        "Bengaluru": "24°C, partly cloudy, humidity 70%",
        "Mumbai":    "32°C, humid, chance of rain 60%",
        "Delhi":     "38°C, sunny, very hot",
        "Chennai":   "34°C, sunny, humidity 80%"
    }
    return weather_data.get(city,f"weather data for {city} not available")

@tool
def get_flight_price(origin:str,destination:str)->str:
    """Search for flight prices between two Indian cities.
    Use when user asks about flights, travel cost, or wants to fly somewhere."""
    prices = {
        ("Bengaluru", "Mumbai"): "₹3,200 - ₹6,500",
        ("Bengaluru", "Delhi"):  "₹4,500 - ₹9,000",
        ("Mumbai", "Delhi"):     "₹3,800 - ₹7,500",
    }
    key = (origin,destination)
    return prices.get(key, f"Prices for {origin} → {destination}: ₹2,500 - ₹8,000 (estimated)")

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount between currencies using live rates.
    Use when user asks about currency conversion or exchange rates."""
    rates = {"INR": 1, "USD": 83.2, "EUR": 89.5, "GBP": 104.8}
    if from_currency not in rates or to_currency not in rates:
        return f"Unsupported currency. Supported: {list(rates.keys())}"
    result = (amount / rates[from_currency]) * rates[to_currency]
    return f"{amount} {from_currency} = {result:.2f} {to_currency}"

tools = [get_weather, get_flight_price, convert_currency]

# step 2: create the agent
llm = ChatGroq(model_name =os.getenv("groq_model_name"))

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a helpful travel assistant. Use the provided tools to answer the user's questions.\n"
        "Rules:\n"
        "1. Call the appropriate tools to gather information.\n"
        "2. Do not repeat the same tool call with the same arguments.\n"
        "3. Once you have the results from the tools, stop calling tools and immediately provide a final, direct answer to the user summarizing the information."
    )),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad") # it stores the intermediate steps (actions and tool outputs) of the agents current execution loop.
])



agent = create_tool_calling_agent(llm,tools,prompt)

excecutor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose = True, # it print the full react loop great for learning
    max_iterations = 5,
    handle_parsing_errors = True
)

# step 3 run it
result = excecutor.invoke({
    "input": "I want to fly from Bengaluru to Delhi. What's the weather there and how much is a ticket?"
})
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'Delhi'}`


38°C, sunny, very hot
Invoking: `get_flight_price` with `{'destination': 'Delhi', 'origin': 'Bengaluru'}`


₹4,500 - ₹9,000The current weather in Delhi is 38°C and sunny. As for the flight price, it ranges from ₹4,500 to ₹9,000.

> Finished chain.
The current weather in Delhi is 38°C and sunny. As for the flight price, it ranges from ₹4,500 to ₹9,000.


- if we keep verbose = True , will see the full react loop printed.
```
> Entering new AgentExecutor chain...
Thought: User wants weather in Delhi and flight prices from Bengaluru to Delhi.
         I should call both tools.

Action: get_weather
Action Input: {"city": "Delhi"}
Observation: 38°C, sunny, very hot

Action: get_flight_price
Action Input: {"origin": "Bengaluru", "destination": "Delhi"}
Observation: ₹4,500 - ₹9,000

Thought: I have all information needed.
Final Answer: Delhi is currently 38°C and very sunny. Flights from Bengaluru 
              to Delhi range from ₹4,500 to ₹9,000.
```

Level 2 — Agent with Memory (Conversational Agent)


In [31]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
load_dotenv(override=True)

session_store = {}

def get_session_history(session_id: str):
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# prompt : Now we including the chat history also
# prompt : Now we including the chat history also
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful travel assistant with memory.
    Remember what the user told you in previous messages.
    
    Rules:
    1. If the user's message is a statement, greeting, or general conversation that does not require information from your tools (weather, flights, currency), respond conversationally and directly without calling any tools.
    2. If you need specific information to answer the user's query, call the appropriate tool.
    3. Do not write any conversational preamble before calling a tool.
    """),
    MessagesPlaceholder("chat_history"), # previous conversation
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad") # react scratchpad
])

agent = create_tool_calling_agent(llm,tools,prompt)
excecutor = AgentExecutor(agent=agent,tools=tools,max_iterations = 10,verbose = True)

# wraping with memory
agent_with_memory = RunnableWithMessageHistory(
    excecutor,
    get_session_history,
    input_messages_key = "input",
    history_messages_key = "chat_history"
)

config = {"configurable":{"session_id":"traveller_Subbu"}}

# multi turn conversation
r1 = agent_with_memory.invoke(
    {"input":"iam planning to travel to Delhi next week."},
    config = config
)
print(r1)
print(f"input: {r1['input']}")
print(f"output: {r1['output']}")
print(f"chat_history:{r1['chat_history']}")

C:\Users\subramani.v\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)




> Entering new AgentExecutor chain...
That's great! Delhi is a wonderful city with a rich history and culture. What are your plans while you're there? Are you looking to explore the city's landmarks, try the local cuisine, or something else?

> Finished chain.
{'input': 'iam planning to travel to Delhi next week.', 'chat_history': [], 'output': "That's great! Delhi is a wonderful city with a rich history and culture. What are your plans while you're there? Are you looking to explore the city's landmarks, try the local cuisine, or something else?"}
input: iam planning to travel to Delhi next week.
output: That's great! Delhi is a wonderful city with a rich history and culture. What are your plans while you're there? Are you looking to explore the city's landmarks, try the local cuisine, or something else?
chat_history:[]


In [33]:
r2 = agent_with_memory.invoke(
    {"input":"What about flights from my city? I'm in Bengaluru"},
    config = config
)
print(f"input: {r2['input']}")
print(f"output: {r2['output']}")
print(f"chat_history:{r2['chat_history']}")



> Entering new AgentExecutor chain...

Invoking: `get_flight_price` with `{'destination': 'Delhi', 'origin': 'Bengaluru'}`


₹4,500 - ₹9,000
Invoking: `get_flight_price` with `{'destination': 'Delhi', 'origin': 'Bengaluru'}`


₹4,500 - ₹9,000The prices for flights from Bengaluru to Delhi range from ₹4,500 to ₹9,000.

> Finished chain.
input: What about flights from my city? I'm in Bengaluru
output: The prices for flights from Bengaluru to Delhi range from ₹4,500 to ₹9,000.
chat_history:[HumanMessage(content='iam planning to travel to Delhi next week.', additional_kwargs={}, response_metadata={}), AIMessage(content="That's great! Delhi is a wonderful city with a rich history and culture. What are your plans while you're there? Are you looking to explore the city's landmarks, try the local cuisine, or something else?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="What about flights from my city? I'm in Bengaluru", additional_k

In [34]:
r3 = agent_with_memory.invoke(
    {"input": "Convert the minimum price to USD for me."},
    config=config
)
print(r3)
print(f"input: {r3['input']}")
print(f"output: {r3['output']}")
print(f"chat_history:{r3['chat_history']}")




> Entering new AgentExecutor chain...

Invoking: `convert_currency` with `{'amount': 4500, 'from_currency': 'INR', 'to_currency': 'USD'}`


4500.0 INR = 374400.00 USD
Invoking: `convert_currency` with `{'amount': 4500, 'from_currency': 'INR', 'to_currency': 'USD'}`


4500.0 INR = 374400.00 USD
Invoking: `convert_currency` with `{'amount': 4500, 'from_currency': 'INR', 'to_currency': 'USD'}`


4500.0 INR = 374400.00 USD
Invoking: `convert_currency` with `{'amount': 4500, 'from_currency': 'INR', 'to_currency': 'USD'}`


4500.0 INR = 374400.00 USD
Invoking: `convert_currency` with `{'amount': 4500, 'from_currency': 'INR', 'to_currency': 'USD'}`


4500.0 INR = 374400.00 USD
Invoking: `convert_currency` with `{'amount': 4500, 'from_currency': 'INR', 'to_currency': 'USD'}`


4500.0 INR = 374400.00 USD
Invoking: `convert_currency` with `{'amount': 4500, 'from_currency': 'INR', 'to_currency': 'USD'}`


4500.0 INR = 374400.00 USD
Invoking: `convert_currency` with `{'amount': 4500, 'from_curre

Level 6 — Agent Control: Safety, Limits, and Human-in-the-Loop


In [ ]:

from langchain_classic.agents import AgentExecutor

# ── Safety controls ───────────────────────────────────────
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=8,           # stop after 8 reasoning steps
    max_execution_time=30,      # stop after 30 seconds
    handle_parsing_errors=True, # don't crash on bad LLM output
    early_stopping_method="generate"  # generate final answer if limit hit
)

# ── Human-in-the-loop for dangerous actions ───────────────
@tool
def delete_customer_account(customer_id: str) -> str:
    """Delete a customer account permanently.
    IMPORTANT: Requires human approval before executing.
    Use ONLY when explicitly instructed AND human confirms."""
    # In production: pause here and wait for human approval
    # via a webhook, Slack message, email, etc.
    human_input = input(f"⚠️  CONFIRM: Delete account {customer_id}? (yes/no): ")
    if human_input.lower() != "yes":
        return f"Account deletion CANCELLED by human operator."
    return f"Account {customer_id} deleted. This action is irreversible."

# ── Structured agent output ───────────────────────────────
from pydantic import BaseModel, Field

class AgentResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question")
    confidence: str = Field(description="high, medium, or low")
    tools_used: list[str] = Field(description="List of tools that were called")
    needs_escalation: bool = Field(description="Whether this needs human follow-up")

structured_llm = llm.with_structured_output(AgentResponse)

# ── Retry on failure ──────────────────────────────────────
from langchain_core.runnables import RunnableLambda

def run_with_fallback(input_data):
    try:
        return executor.invoke(input_data)
    except Exception as e:
        return {
            "output": f"I encountered an issue: {str(e)}. Please try rephrasing your question.",
            "error": str(e)
        }

safe_executor = RunnableLambda(run_with_fallback)